# disent_from_attn_out — disentangle q / kv transfer from o-transfer

**Problem.** Our standard q-transfer and kv-transfer setups both make DST pursue `SRC.targ` — but so
does an **o-transfer** (transplanting the goal heads' ATTN-OUT / OV contribution, i.e. the *output* of
the DIFF read). So the standard setups can't tell the three pathways apart: q, kv, and o all point at
`SRC.targ`.

**Fix.** Design setups where the three pathways predict **different** tokens.

| section | q / kv transfer → | o-transfer → | clean DST → |
|---|---|---|---|
| **q-transfer** | **T4** (progress pointer resolved against DST's *own* list) | **T3** (`SRC.targ`, resolved goal) | T2 |
| **kv-transfer** | **T2** (SRC task *menu* resolved at DST's progress) | **T3** (`SRC.targ`) | T4 |

- **q-transfer.** SRC(T1,T2,T3), prefill{T1,T2}, targ **T3**;  DST(T1,T2,T4), prefill{T1}, clean-targ **T2**.
  If the query carries *progress* ("T1,T2 done"), transplanting it into DST (which has only T1 done)
  should make DST skip to **T4** (its own 3rd slot). The OV output carries the resolved goal → **T3**.
- **kv-transfer.** SRC(T1,T2,T3), prefill{T1,T2}, targ **T3**;  DST(T1,T4,T5), prefill{T1}, clean-targ **T4**.
  If the keys/values carry the *task list* {T1,T2,T3}, transplanting it into DST (T1 done) should make
  DST do **T2** (the SRC menu's next-after-T1). The OV output → **T3**.

Since **T4 ∉ SRC** and **T2 ≠ T3**, each pathway has a unique signature. Qualitative check only.

Utils exercised: pad-aligned k/v span transplant (kv), attn-gaze debugging (q), GoalScope readout (both),
and a small o_proj-input replacement helper for o-transfer.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0, 1")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append("../..")
import torch, numpy as np, baukit
from contextlib import contextmanager, nullcontext
import src.diff.functional as FUNC
from src.diff.functional import *          # public helpers into scope
import src.diff.data as DATA
from src.attention import get_attention_matrices
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape

FUNC.setup("Qwen/Qwen2.5-Coder-32B-Instruct")  # loads model + head masks (once)
mt = FUNC.mt

2026-09-11 20:07:32 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [2]:
# OPTIONAL — use the top-250 goal-head set (patch + readout), which gave near-perfect kv transfer.
import json
_p = "../../head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json"
GOAL_HEADS_250 = [tuple(h) for h in json.load(open(_p))]

In [3]:
################################################################################################################################
GOAL_HEADS, UNION_HEADS, GROUP = FUNC.GOAL_HEADS, FUNC.UNION_HEADS, FUNC.GROUP
PATCH_HEADS = GOAL_HEADS_250                     # heads carrying the TRANSPLANT (q / kv / o / q-freeze). top-250: next cell.
READOUT_HEADS = GOAL_HEADS_250                   # heads read by GoalScope + the ATTN-viz panels (independent of PATCH_HEADS)
ALPHA = 2.0                                  # OV-write boost of PATCH_HEADS at the pivot on the PATCHED gen/readout (1.0 = off)
O_ALPHA = 1.0                                # o-transfer (GoalScope 'replace') source-side weight (1.0 = full replace)

print(f"PATCH_HEADS = {len(PATCH_HEADS)} · READOUT_HEADS = {len(READOUT_HEADS)}")
################################################################################################################################


# ── OVReader (layerwise OV-write read-out) config — mirrors 07.3 ──
from contextlib import ExitStack
from src.goals.goalscope import OVReader
from scripts.evaluation.eval_goalscope import CAUSAL_PREFIX
from src.tokens import prepare_input

MODE            = "faithfulness"           # faithfulness | causality
OV_PROBE_LAYERS = list(range(36, 62, 2))   # residual "after layer L" probed by OVReader
STEER_BAND      = (35, 46)                 # GoalScope steer band for the read-out
ALPHA_OV        = 0.15                      # OVReader read-out strength (additive_norm)
MAXNEW          = 64                        # OVReader read-out length
SHOW_SRC        = True                     # also OVReader the clean SRC run
SHOW_CLEAN_DST  = True                     # also OVReader the clean DST run

COLLAPSE_LAYER  = 40                       # depth-collapse: force ALL goal heads to read residual h^L
ov     = OVReader(mt, READOUT_HEADS)        # rebuild this cell if READOUT_HEADS changes
PREFIX = CAUSAL_PREFIX if MODE == "causality" else ""
print(f"OVReader ready · probe layers {OV_PROBE_LAYERS} · mode={MODE}")

PATCH_HEADS = 250 · READOUT_HEADS = 250


Building OV readers:   0%|          | 0/250 [00:00<?, ?it/s]

OVReader ready · probe layers [36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60] · mode=faithfulness


In [4]:
# ── helpers (prompt display, instance builder, o-transfer via o_proj-input replacement) ──
def pivot_of(prompt, neg_shift=-1):
    tok = prepare_input(prompts=prompt, tokenizer=mt) if isinstance(prompt, str) else prompt
    return tok["input_ids"].shape[1] + neg_shift

def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)),
                     title=f"[bold]{escape(title)}[/bold]", title_align="left",
                     border_style="grey42", padding=(1, 1)))

BANK = DATA.QTASKS["list[str]"]; BY = {p.id: p for p in BANK.predicates}
# five distinct, pairwise-compatible list[str] predicates
A, B, C, D, E = "total_length", "total_uppercase_letters", "has_palindrome", "count_starts_vowel", "count_with_digit"

NAMING = "letters"

class _SeqQTI(DATA.QTransferInstance):
    """Notebook-local: name result vars sequentially by APPEARANCE in the prefill (progress order,
    then the frontier target), so prefilled lines read a, b, c, ... regardless of each predicate's
    index in the task list. Only affects rendered var names; task-list order & token positions are
    unchanged. Assumes src_progress == dst_progress (mk sets them equal)."""
    def var_of(self, pred_id):
        if self.naming != "letters":
            return super().var_of(pred_id)
        order = list(self.src_progress) + [self.expected_next]
        for pr in self.task.predicates:
            if pr.id not in order:
                order.append(pr.id)
        return chr(ord("a") + order.index(pred_id))

def mk(pred_ids, progress, target, lang="python"):
    """A QTransferInstance over an arbitrary predicate list (listed presentation, letters naming).
    `lang` sets the instance language; we only read src_parts() of a SRC-instance or dst_parts() of a
    DST-instance, so pass lang=SRC_LANG for the SRC side and lang=DST_LANG for the DST side."""
    task = DATA.dataclasses.replace(BANK, id="xtask", predicates=tuple(BY[i] for i in pred_ids))
    return _SeqQTI(task=task, var_name=BANK.var_names[0], presentation="listed",
        src_lang=lang, dst_lang=lang, src_progress=progress, dst_progress=progress,
        expected_next=target, naming=NAMING, meta={"datatype": "list[str]"})

def list_body(inst):
    """The numbered task-list body exactly as _question renders it (the kv transplant span)."""
    return "\n".join(f"({i+1}) {p.description}" for i, p in enumerate(inst.task.predicates))

def ov_boost(positions=(-1,)):
    """OV-write boost of PATCH_HEADS by ALPHA at `positions` (nullcontext when ALPHA==1.0)."""
    return boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=positions) if ALPHA != 1.0 else nullcontext()

# ── o-transfer = GoalScope strategy="replace", PERSISTENT across ALL decode steps ──
# The EXACT GoalScope replace intervention: replace PATCH_HEADS' o_proj INPUT (per-head attn output)
# at the destination's last token with SRC's (extracted at the SRC pivot), re-applied EVERY generation
# step (n_steer_tokens="all"). We point the GoalScope dest at the DST *code* prompt, so it generates
# DST's continuation. O_ALPHA = replace source-side weight (1.0 = full replace).
from src.goals.goalscope import apply_goalscope

def o_transfer_gen(src_prompt, dst_prompt, src_pivot, heads, alpha=None, max_new=96):
    a = O_ALPHA if alpha is None else alpha
    res = apply_goalscope(prompt=src_prompt, mt=mt, selected_heads=list(heads), strategy="replace",
                          alpha=a, goalscope_prompt=dst_prompt, token_positions=[src_pivot],
                          n_steer_tokens="all", max_new_tokens=max_new, show=False)
    return res[0]["patched_gen"]

def show_spans(tok, title, task_span=None, kv_span=None, piv=-1):
    """Token-aware prompt renderer (for pad-aligned TokenizerOutputs). task_span -> bright_yellow;
    kv_span -> black on dark_orange (the EXACT transplant window); pivot -> salmon. Title is escaped."""
    ids = (prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok)["input_ids"][0].tolist()
    n = len(ids); pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set()
    kv = set(range(*kv_span)) if kv_span else set()
    PO, PC, KO, KC, TO, TC = "\x00<", "\x00>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos:   s = PO + s + PC
        elif i in kv:     s = KO + s + KC
        elif i in task:   s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    txt = (txt.replace(PO, "[bold black on salmon1]").replace(PC, "[/bold black on salmon1]")
              .replace(KO, "[bold black on dark_orange]").replace(KC, "[/bold black on dark_orange]")
              .replace(TO, "[bright_yellow]").replace(TC, "[/bright_yellow]"))
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

# NOTEBOOK-LOCAL get_prompt shadow: plain pass-through to the module builder (no instruction tweak).
# base = the MODULE function FUNC.get_prompt (always unwrapped) -> safe to re-run this cell (no recursion).
def get_prompt(question, prefill, ctx=None):
    return FUNC.get_prompt(question, prefill, ctx=ctx)

def stub_parts(inst, side, stub):
    """(question, prefill) honoring the pivot `stub`. 'indent' = native trailing-indent pivot; 'var_eq'
    = end at the frontier's '{var} =' (py) / 'auto {var} =' (cpp), with shared cpp result decls rewritten
    to auto — matches the scaled-up MultiPredicateInstance var_eq. Works for q- and kv-transfer instances."""
    q, pf = (inst.src_parts() if side == "src" else inst.dst_parts())
    if stub != "var_eq":
        return q, pf
    lang = inst.src_lang if side == "src" else inst.dst_lang
    prog = inst.src_progress if side == "src" else inst.dst_progress
    var = inst.var_of(inst.expected_next)                 # the frontier's positional variable
    body = pf.rstrip("\n ")                                # drop the trailing indent line
    if lang == "cpp":
        for pid in prog:
            body = DATA._cpp_autoify(body, inst.var_of(pid))  # shared decls -> auto (match var_eq)
        return q, body + f"\n    auto {var} ="
    return q, body + f"\n    {var} ="
print("helpers ready ·", {k: BY[k].description for k in (A,B,C,D,E)})


# ── OVReader layerwise helpers (mirrors 07.3): residual after each probe layer -> OV-write read-out ──
def extract_resid(tok, pivot, layers, patch_stack=None):
    """Residual after each layer at `pivot`, optionally under a patch ExitStack (e.g. the q-transfer)."""
    tok = prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok
    ln  = lambda l: mt.layer_name_format.format(l)
    unt = lambda x: x[0] if isinstance(x, tuple) else x
    ctx = patch_stack if patch_stack is not None else nullcontext()
    with torch.no_grad(), ctx, baukit.TraceDict(mt._model, [ln(l) for l in layers], retain_output=True) as tr:
        mt._model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
    return {l: unt(tr[ln(l)].output)[0, pivot].detach().float().cpu() for l in layers}

def ov_layerwise(resid_by_layer):
    """Print the full OVReader read-out per probe layer (one batched apply_probe over all layers)."""
    outs = ov.apply_probe(resid_by_layer, mode=MODE, steer_layers=list(range(*STEER_BAND)),
                          max_new_tokens=MAXNEW, alpha=ALPHA_OV)
    for L in OV_PROBE_LAYERS:
        print(f"    OVReader L={L:2d}: {escape(PREFIX + outs[L])}")


# ── depth-collapse tools: force ALL goal heads to read the SAME residual h^L (after layer L) ──
# Instead of each head reading its own layer's residual, overwrite every head-layer's BLOCK INPUT with
# the cached h^L in one forward, so the model itself applies input_layernorm (RMSNorm) + RoPE. Tool 1
# reads the value-weighted pivot->task attention under this collapse (does the query already know what
# it wants by layer L?); Tool 2 reads the collective OV write and verbalizes it (goalscope_readout).
from contextlib import contextmanager as _cm

def resid_all(tok, layer):
    """Residual after `layer` at EVERY position -> (seq, n_embd) CPU f32 (one clean forward)."""
    tok = prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok
    ln  = mt.layer_name_format.format(layer)
    unt = lambda x: x[0] if isinstance(x, tuple) else x
    with torch.no_grad(), baukit.TraceDict(mt._model, [ln], retain_output=True) as tr:
        mt._model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
    return unt(tr[ln].output)[0].detach().float().cpu()

@_cm
def read_from_layer(h_L, heads):
    """Force each layer in `heads` to read residual `h_L` (seq, n_embd) as its block input, this forward.
    Guarded by seq-length match, so it is a no-op on forwards of a different length (e.g. the probe gen)."""
    ref = next(mt._model.parameters()); dev, dt = ref.device, ref.dtype
    hL = h_L.to(dev, dt)                                  # (seq, n_embd)
    handles = []
    try:
        for L in sorted({l for l, _ in heads}):
            blk = baukit.get_module(mt._model, mt.layer_name_format.format(L))
            def pre_hook(module, args, kwargs):
                hs = args[0] if args else kwargs.get("hidden_states")
                if hs is None or hs.shape[1] != hL.shape[0]:
                    return None                          # length mismatch -> leave untouched
                new = hL.unsqueeze(0).expand(hs.shape[0], -1, -1).to(hs.dtype).to(hs.device)
                if args:
                    return (new,) + tuple(args[1:]), kwargs
                kwargs = dict(kwargs); kwargs["hidden_states"] = new
                return args, kwargs
            handles.append(blk.register_forward_pre_hook(pre_hook, with_kwargs=True))
        yield
    finally:
        for h in handles:
            h.remove()

def read_from_layer_iface(h_L, heads):
    """patch_interface form of read_from_layer, for get_attention_matrices / goalscope_readout."""
    def iface(mt, inputs, patches=None, model_kwargs=None):
        with read_from_layer(h_L, heads):
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return iface

# *** Tool 1: value-weighted pivot->task attention when all `heads` read h^L (vs clean native depth) ***
def collapsed_gaze(tok, pivot, task_positions, layer=None, heads=None, show_clean=True):
    layer = COLLAPSE_LAYER if layer is None else layer
    heads = list(READOUT_HEADS if heads is None else heads)
    tok   = prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok
    iface = read_from_layer_iface(resid_all(tok, layer), heads)
    toks  = [mt.tokenizer.decode([int(tok["input_ids"][0][j])]) for j in task_positions]
    a_coll = pivot_attn(get_attention_matrices(tok, mt, value_weighted=True, patch_interface=iface),
                        heads, task_positions, pivot)
    if show_clean:
        a_clean = pivot_attn(get_attention_matrices(tok, mt, value_weighted=True),
                             heads, task_positions, pivot)
        show_delta_tokens(toks, a_clean,          f"gaze . CLEAN native depth (pivot->task)")
        show_delta_tokens(toks, a_coll,           f"gaze . COLLAPSED all heads read h^{layer} (pivot->task)")
        show_delta_tokens(toks, a_coll - a_clean, f"gaze Delta . collapsed(h^{layer}) - clean")
    else:
        show_delta_tokens(toks, a_coll,           f"gaze . COLLAPSED all heads read h^{layer} (pivot->task)")
    return a_coll

# *** Tool 2: verbalize the collective OV write when all `heads` read h^L (goalscope under the collapse) ***
def collapsed_readout(tag, prompt, pivot, layer=None, heads=None):
    layer = COLLAPSE_LAYER if layer is None else layer
    heads = list(READOUT_HEADS if heads is None else heads)
    tok   = prepare_input(prompts=prompt, tokenizer=mt) if isinstance(prompt, str) else prompt
    iface = read_from_layer_iface(resid_all(tok, layer), heads)
    return goalscope_readout(f"{tag} . COLLAPSED h^{layer}", prompt, pivot, heads=heads, iface=iface)

print("depth-collapse tools ready · collapsed_gaze(...) · collapsed_readout(...) · COLLAPSE_LAYER =", COLLAPSE_LAYER)

helpers ready · {'total_length': 'the total number of characters across all words', 'total_uppercase_letters': 'the total number of uppercase letters across all words', 'has_palindrome': 'whether any word is a palindrome', 'count_starts_vowel': 'the number of words that start with a vowel', 'count_with_digit': 'the number of words that contain a digit'}


## 1. q-transfer  (query ⇒ progress)  — disentangle from o-transfer

SRC(T1,T2,T3) prefill{T1,T2} → about to write **T3**.  DST(T1,T2,T4) prefill{T1} → about to write **T2**.

- **query transplant** should trigger **T4** (SRC's progress "T1,T2 done" resolved against DST's own list).
- **o-transfer** (goal-head attn-out) should trigger **T3** (the resolved SRC goal).
- clean DST writes **T2**.

In [5]:
# build SRC / DST prompts for the q-transfer disentanglement
Q_STUB = "var_eq"                            # q-transfer pivot: "indent" (scaled-up default) | "var_eq"
SRC_LANG, DST_LANG = "python", "python"   # per-side language; e.g. "python"/"cpp" for cross-lang q-transfer

# q_src = mk([A, B, C, D], [A, B], C, lang=SRC_LANG)
# q_dst = mk([A, B, D, C], [A], B, lang=DST_LANG) 

q_src = mk([D, A, B, C], [A, C, B], D, lang=SRC_LANG)
q_dst = mk([A, C, B, E, D], [A], C, lang=DST_LANG) 

# q_src = mk([D, B, C], [C, B], D, lang=SRC_LANG)
# q_dst = mk([C, B, E, D], [], C, lang=DST_LANG) 

q_src = mk([D, A, B, C], [A, C, B], D, lang=SRC_LANG)
q_dst = mk([A, C, B, E], [A], C, lang=DST_LANG) 

_q, _pf = stub_parts(q_src, "src", Q_STUB); qsrc_prompt = get_prompt(_q, DATA._finalize_prefill(_pf))
_q, _pf = stub_parts(q_dst, "dst", Q_STUB); qdst_prompt = get_prompt(_q, DATA._finalize_prefill(_pf))
qpiv_src, qpiv_dst = pivot_of(qsrc_prompt), pivot_of(qdst_prompt)
print(f"T2={BY[B].description!r}  T3={BY[C].description!r}  T4={BY[D].description!r}")
show(qsrc_prompt, f"SRC · [{A},{B},{C}] · {{{A},{B}}} done · targ {C} · pivot@{qpiv_src}")
show(qdst_prompt, f"DST · [{A},{B},{D}] · {{{A}}} done · clean-targ {B} · pivot@{qpiv_dst}")

qsrc_tok = prepare_input(prompts=qsrc_prompt, tokenizer=mt)
qdst_tok = prepare_input(prompts=qdst_prompt, tokenizer=mt)

T2='the total number of uppercase letters across all words'  T3='whether any word is a palindrome'  T4='the number of words that start with a vowel'


╭─ SRC · [total_length,total_uppercase_letters,has_palindrome] · {total_length,total_uppercase_letters} done · ta─╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words that start with a vowel\n                                                               │
│ (2) the total number of characters across all words\n                                                           │
│ (3) the total number of uppercase letters across all words\n                                                    │
│ (4) whether any word is a palindrome\n                                                                          │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(len(w) for w in words)\n                                                                            │
│     b = any(w == w[::-1] for w in words)\n                                                                      │
│     c = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     d =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · [total_length,total_uppercase_letters,count_starts_vowel] · {total_length} done · clean-targ total_upper─╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the total number of characters across all words\n                                                           │
│ (2) whether any word is a palindrome\n                                                                          │
│ (3) the total number of uppercase letters across all words\n                                                    │
│ (4) the number of words that contain a digit\n                                                                  │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(len(w) for w in words)\n                                                                            │
│     b =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [6]:
# clean baselines + GoalScope readouts
rich_print(f"[red1]clean SRC (expect {C!r}):[/red1]\n{escape(greedy(qsrc_tok, max_new=96))}")
goalscope_readout("   ↳ goalscope · SRC", qsrc_prompt, qpiv_src, heads=READOUT_HEADS)
# ── OVReader layerwise on the CLEAN runs (SHOW_SRC / SHOW_CLEAN_DST) ──
if SHOW_SRC:
    rich_print("[red1]   ↳ OVReader layerwise · clean SRC[/red1]")
    ov_layerwise(extract_resid(qsrc_tok, qpiv_src, OV_PROBE_LAYERS))

rich_print(f"[sea_green3]clean DST (expect {B!r}):[/sea_green3]\n{escape(greedy(qdst_tok, max_new=96))}")
goalscope_readout("   ↳ goalscope · DST", qdst_prompt, qpiv_dst, heads=READOUT_HEADS)
if SHOW_CLEAN_DST:
    rich_print("[sea_green3]   ↳ OVReader layerwise · clean DST[/sea_green3]")
    ov_layerwise(extract_resid(qdst_tok, qpiv_dst, OV_PROBE_LAYERS))

clean SRC (expect 'has_palindrome'):
 sum(1 for w in words if w[0].lower() in 'aeiou')
    return d, a, c, b
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


   ↳ goalscope · SRC  pos=174 ' =' => I'm checking if the variable starts with a vowel.

   ↳ OVReader layerwise · clean SRC

    OVReader L=36: I'm checking if there's a character that appears more than once in the string, ignoring case sensitivity.
    OVReader L=38: I'm checking if an element is in a set of unique elements.
    OVReader L=40: Identifying if any character in the string is a vowel.
    OVReader L=42: Identifying variables starting with a vowel (e.g., 'a', 'e', 'i', 'o', 'u') in a list of words.
    OVReader L=44: I'm checking if any element in the list matches the condition of starting with a vowel (e.g., 'a', 'e', 'i', 'o', 'u').
    OVReader L=46: I'm checking if the string starts with any vowel (a, e, i, o, u).
    OVReader L=48: I'm checking how many lines start with a word that matches one of the specified patterns (e.g., lines starting with "The" or "the", where "The" is considered a count of lines starting with words like "The" or "the" for those who start with a letter matching one of the specified
    OVReader L=50: I'm counting lines that start with a specific character, such as 'a

clean DST (expect 'total_uppercase_letters'):
 any(w == w[::-1] for w in words)
    c = sum(sum(1 for ch in w if ch.isupper()) for w in words)
    d = sum(any(ch.isdigit() for ch in w) for w in words)
    return a, b, c, d
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · DST  pos=139 ' =' => I'm checking if any string in the list is a palindrome.

   ↳ OVReader layerwise · clean DST

    OVReader L=36: I'm checking if there's at least one element that satisfies a condition (usually a boolean function) in a list or string, using Python's `any()` function with a lambda function to check for duplicates.
    OVReader L=38: I'm checking if any element in a list is true, using the `any()` function with a generator expression.
    OVReader L=40: I'm checking if any element in the list is True when applying the `any()` function with a generator expression that checks for any element where the condition is met.
    OVReader L=42: I'm checking if any element in the list is true when evaluated with a condition that checks for truthiness, specifically using the `any()` function with a generator expression.
    OVReader L=44: I'm checking if any element in a list is true when applying a function that checks for truthiness with `any(map(lambda x: x, my_list))` or simply `any(x for x in my_list if x)` if using Python 3.8+ with the walrus operator.
    OVReader L=46: I'm checking 

In [7]:
len(PATCH_HEADS)

250

In [8]:
# (A) q-transfer: transplant SRC pivot QUERY → DST pivot   [expect T4]
qQ_LAYERS = sorted({l for l, _ in PATCH_HEADS})
q_cache = cache_src_proj(qsrc_prompt, [qpiv_src], qQ_LAYERS, "q")
_qh = heads_per_layer(PATCH_HEADS, "q")
qpf = [dict(which="q", cache=q_cache, heads=_qh, positions=[qpiv_dst])]
qdc = [dict(which="q", cache=q_cache, heads=_qh, positions=[-1])]
q_iface = make_patch_interface(mt, qpf)                 # PLAIN patch (used for gaze — boost scales the write, not attention)
with ov_boost():
    qgen = greedy(qdst_tok, prefill_specs=qpf, decode_specs=qdc, n_steer=32, max_new=256)
def q_patched_iface():                                   # query patch (+ ALPHA boost) for the readout
    if ALPHA == 1.0:
        return q_iface
    def _if(mt, inputs, patches=None, model_kwargs=None):
        with proj_patch_hooks(mt, qpf), ov_boost():
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return _if
rich_print(f"[salmon1]q-transfer · DST (expect T4={D!r}):[/salmon1]\n{escape(qgen)}")
goalscope_readout("   ↳ goalscope · q-transfer", qdst_prompt, qpiv_dst, heads=READOUT_HEADS, iface=q_patched_iface())

# gaze debugging: value-weighted pivot→DST-TASK attention, clean vs q-patched (SRC target T3 not in DST)
q_spans = [token_span(qdst_prompt, p.description) for p in q_dst.task.predicates]
qTASK = list(range(min(s[0] for s in q_spans), max(s[1] for s in q_spans)))
qtoks = [mt.tokenizer.decode([int(qdst_tok["input_ids"][0][j])]) for j in qTASK]
q_clean_i = get_attention_matrices(qdst_tok, mt, value_weighted=True)
q_patch_i = get_attention_matrices(qdst_tok, mt, value_weighted=True, patch_interface=q_iface)
a_c = pivot_attn(q_clean_i, list(READOUT_HEADS), qTASK, qpiv_dst)
a_p = pivot_attn(q_patch_i, list(READOUT_HEADS), qTASK, qpiv_dst)
show_delta_tokens(qtoks, a_c,       "gaze · CLEAN   pivot→DST-TASK (value-weighted, READOUT_HEADS)")
show_delta_tokens(qtoks, a_p,       "gaze · PATCHED pivot→DST-TASK (value-weighted, READOUT_HEADS)")
show_delta_tokens(qtoks, a_p - a_c, "gaze Δ · patched − clean (pivot→DST-TASK, READOUT_HEADS)")


# ── OVReader layerwise on the PATCHED DST run (under the q-transfer intervention) ──
def q_patch_stack():
    """ExitStack of the exact q-transfer intervention: query patch at the pivot + OV boost."""
    es = ExitStack()
    es.enter_context(proj_patch_hooks(mt, qpf))
    es.enter_context(ov_boost())
    return es
rich_print(f"[salmon1]   ↳ OVReader layerwise · q-transfer (patched DST)[/salmon1]")
ov_layerwise(extract_resid(qdst_tok, qpiv_dst, OV_PROBE_LAYERS, q_patch_stack()))

q-transfer · DST (expect T4='count_starts_vowel'):
 any(any(w.isalpha() and w == w.lower() for w in words) for w in words)
    c = sum(any(w.isalpha() and w == w.upper() for w in words) for w in words)
    d = sum(any(any(char.isdigit() for char in w) for w in words))
    return a, b, c, d
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · q-transfer  pos=139 ' =' => I'm checking if any string in the list has at least one character that
is a digit.

gaze · CLEAN   pivot→DST-TASK (value-weighted, READOUT_HEADS)   (green +Δ / red −Δ, |Δ|max=0.714)

gaze · PATCHED pivot→DST-TASK (value-weighted, READOUT_HEADS)   (green +Δ / red −Δ, |Δ|max=0.356)

gaze Δ · patched − clean (pivot→DST-TASK, READOUT_HEADS)   (green +Δ / red −Δ, |Δ|max=0.407)

   ↳ OVReader layerwise · q-transfer (patched DST)

    OVReader L=36: I'm checking if there's at least one character that appears more than once in a string by using a set to track unique characters.
    OVReader L=38: I'm checking if any element in the list is True, where each element is a boolean indicating whether an item is a palindrome.
    OVReader L=40: I'm checking if any element in the list is true when each element is transformed into a boolean by applying a lambda function that checks for non-empty strings with all characters being alphabetic and returns True if at least one string is entirely alphabetic.
    OVReader L=42: I'm checking if there's any element in the list that is a palindrome by using a generator expression with `any` function, where the element is equal to its reverse when reversed with a boolean check.
    OVReader L=44: I'm checking if any element in a list matches a condition using a generator expression with `any()`.
    OVReader L=46: I'm checking if any string in a list has a substring that is also a s

## 1b. depth-collapse — all goal heads read the SAME residual h^L

`collapsed_gaze` (Tool 1): by layer `COLLAPSE_LAYER`, does the pivot query already point at the right task tokens? `collapsed_readout` (Tool 2): verbalize the collective OV write those heads produce reading h^L. Both force every goal head onto h^L (correct RMSNorm+RoPE via a block-input overwrite); complementary to the OVReader-layerwise cells, which assume full attention to the pivot.

In [ ]:
# ── depth-collapse demo on the q-transfer SRC / DST prompts (edit COLLAPSE_LAYER above to sweep) ──
def _task_positions(prompt, inst):
    spans = [token_span(prompt, pr.description) for pr in inst.task.predicates]
    return list(range(min(s[0] for s in spans), max(s[1] for s in spans)))

for tag, prompt, tok, piv, inst in [("SRC", qsrc_prompt, qsrc_tok, qpiv_src, q_src),
                                    ("DST", qdst_prompt, qdst_tok, qpiv_dst, q_dst)]:
    rich_print(f"[bold]=== depth-collapse @ L={COLLAPSE_LAYER} · {tag} (pivot@{piv}) ===[/bold]")
    collapsed_gaze(tok, piv, _task_positions(prompt, inst))
    collapsed_readout(f"   ↳ collapsed readout · {tag}", prompt, piv)
